In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Libraries 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Preprocess of the data
import re
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from scipy.sparse import csr_matrix, hstack

# Transformers
from sklearn.base import BaseEstimator, TransformerMixin

# Pipeline
from sklearn.pipeline import Pipeline

# Model: Split, Traing, Metrics
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Save and Load trained Model
import joblib


In [ ]:
# Import Datasets
df_train=pd.read_csv('/kaggle/input/nlp-getting-started/train.csv', index_col="id")
df_test=pd.read_csv('/kaggle/input/nlp-getting-started/test.csv', index_col="id")

# Functions

In [ ]:
#----------------------------------------------------------------------------------------------
def clean_text(text):
    
    # Lowercase all the text
    text = text.lower()
    # Delete URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Delete punctuation marks
    text = re.sub(r'[^\w\s]', '', text)
    # Delete numbers (optional)
    text = re.sub(r'\d+', '', text)
    return text
    
#----------------------------------------------------------------------------------------------
def tokenizator(df):
    
    #nltk.download('punkt')
    #nltk.download('punkt_tab')
    
    # Tokenizer
    df['tokens'] = df['clean_text'].apply(word_tokenize)
    # Create  new column with the tokens
    #df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))
    # View results
    #df[['clean_text','tokens','processed_text']].head()
    return df
    
#----------------------------------------------------------------------------------------------
def stop_word(df):
    
    #nltk.download('stopwords')

    stop_words = set(stopwords.words('english'))
    # Delete stopwords
    df['tokens'] = df['tokens'].apply(lambda x: [w for w in x if w not in stop_words])
    # View results
    #df[['clean_text','tokens']].head()
    return df
    
#----------------------------------------------------------------------------------------------
def lemmatizer(df):
    
    #nltk.download('wordnet')
    #nltk.download('omw-1.4')
    
    lemmatizer = WordNetLemmatizer()
    # Apply of lemmatizer
    df['tokens'] = df['tokens'].apply(lambda x: [lemmatizer.lemmatize(w) for w in x])
    # View results
    #df[['clean_text','tokens']].head()
    return df
    
#----------------------------------------------------------------------------------------------
class TextPreprocessor(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        df = X.copy()
        df['text_length'] = df['text'].apply(len)
        df['clean_text'] = df['text'].apply(clean_text)
        df['word_count'] = df['text'].apply(lambda x: len(x.split()))
        df['num_exclamations'] = df['text'].apply(lambda x: x.count('!'))
        df['num_uppercase'] = df['text'].apply(lambda x: sum(1 for c in x if c.isupper()))
        df['num_questions'] = df['text'].apply(lambda x: x.count('?'))
        df = tokenizator(df)
        df = stop_word(df)
        df = lemmatizer(df)
        df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))
        return df

#----------------------------------------------------------------------------------------------
class FeatureCombiner(BaseEstimator, TransformerMixin):
    
    def __init__(self, tfidf, bert_model):
        self.tfidf = tfidf
        self.bert_model = bert_model

    def fit(self, X, y=None):
        self.tfidf.fit(X['processed_text'])
        return self
    
    def transform(self, X, y=None):
        # TF-IDF
        X_tfidf = self.tfidf.transform(X['processed_text'])
        
        # Embeddings
        X_embeddings = self.bert_model.encode(X['processed_text'].tolist())
        X_embeddings_sparse = csr_matrix(X_embeddings)

        # Engineered features
        cols = ['text_length','word_count','num_exclamations','num_uppercase','num_questions']
        X_features = X[cols].to_numpy()
        X_features_sparse = csr_matrix(X_features)

        # Concatenate
        X_combined = hstack([X_tfidf, X_features_sparse, X_embeddings_sparse])
        return X_combined
    
#----------------------------------------------------------------------------------------------
def output(df, pipeline):
    
    y_pred = pipeline.predict(df)
    output=pd.DataFrame({'id':df.index, 'y':y_pred})
    output.to_csv('submission.csv', index=False)
    output.head()
    return output
    

# Copy of the df_train for not change the Original dataset

In [ ]:
df_model = df_train.copy()

# Visualization of the data
print(20*'****')
print('DIMENSION OF THE DATA IMPORTED: \n')
print('df_model  = ',df_model.shape)
print(20*'****')
print('df_model: \n')
print(df_model.head(),'\n')
print(20*'****')
print('df_model COLUMNS: \n\nCOLUMN  |||  DTYPE  |||  UNIQUE VALUES\n')
for i in df_model.columns:
    print(f'"{i}"  |||  dtype:{df_model[i].dtypes}  |||  unique_values:{df_model[i].nunique()}\n')
print(20*'****')
print('df_model DESCRIBE:')
print(df_model.describe())
print(20*'****')

# Split of the data

In [ ]:
X=df_model.drop(["target"], axis=1)
y=df_model.target

# y -> tu columna objetivo (ej: 'target')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)


# Pipeline

In [ ]:
# Cargas tus modelos previamente entrenados
bert_model = SentenceTransformer('distilbert-base-nli-mean-tokens')
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# Pipeline
nlp_pipeline = Pipeline([
    ('preprocessor', TextPreprocessor()),  
    ('features', FeatureCombiner(tfidf, bert_model)),  
    ('clf', LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='saga',
        class_weight='balanced',
        max_iter=2000,
        random_state=42,
        n_jobs=-1
    ))  
])

# .fit of Pipeline
Log_Reg_model = nlp_pipeline.fit(X_train, y_train)

# Predictions
y_pred = Log_Reg_model.predict(X_test)
y_pred_proba = Log_Reg_model.predict_proba(X_test)[:, 1]

# --- Métrics ---
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nFull report:")
print(classification_report(y_test, y_pred))


In [ ]:
# Submission
output(df_test, Log_Reg_model)


# Save - Load Model

In [ ]:
"""
# Save the trained model
joblib.dump(Log_Reg_model, "Log_Reg_pipe_model.pkl")

# Load the trained model
loaded_model = joblib.load("Log_Reg_pipe_model.pkl")

# Use it...
y_pred = loaded_model.predict(X_test)
"""